# Deep Hedging - Testing & Ideas

### Ideas

- Transaction costs
- Deep hedging (batches with different models (Heston, G2++, GARCH, etc.))
- Other risk measure (for optimizing final pnl)
- Time dependant risk measure (risk managed paths pnl)

In [21]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import norm

In [22]:
class DeepHedging(nn.Module):
    """
    Feed forward neural network with 2 hidden layers.
    - 3 inputs: log-moneyness, time to maturity, previous delta δ_{t-1}
    - Activation function: ReLU
    - Output activation function: Sigmoid (ensures delta in [0,1])
    """
    def __init__(self, K, r, T, kappa=0.0, num_features=3, hidden_size=64):
        super().__init__()
        self.K = K
        self.r = r
        self.T = T
        self.kappa = kappa
        self.num_features = num_features
        self.hidden_size = hidden_size

        self.net = nn.Sequential(
            nn.Linear(num_features, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, 1), nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

    def build_features(self, S):
        """Precompute time-invariant features: log-moneyness and ttm."""
        N_paths, N_steps_plus1 = S.shape
        N_steps = N_steps_plus1 - 1
        log_moneyness = torch.log(S[:, :-1] / self.K)             # (N_paths, N_steps)
        ttm = torch.linspace(self.T, 0, N_steps + 1)[:-1]         # (N_steps,)
        ttm_grid = ttm.unsqueeze(0).expand(N_paths, -1)            # (N_paths, N_steps)
        
        return log_moneyness, ttm_grid, bs_delta

    def compute_deltas(self, S):
        """
        Sequential delta computation: at each step t, feed (log-moneyness_t, ttm_t, δ_{t-1})
        so the NN can explicitly condition on the previous position when deciding to rebalance.
        δ_{t-1} is detached to avoid backprop through the sequential dependency (efficiency).
        """
        N_paths, N_steps_plus1 = S.shape
        N_steps = N_steps_plus1 - 1
        log_moneyness, ttm_grid = self.build_features(S)

        delta_prev = torch.zeros(N_paths)                          # initial position is 0
        deltas = []
        for t in range(N_steps):
            feat = torch.stack([log_moneyness[:, t],
                                ttm_grid[:, t],
                                delta_prev], dim=1)                # (N_paths, 3)
            delta_t = self(feat).squeeze(1)                        # (N_paths,)
            deltas.append(delta_t)
            delta_prev = delta_t.detach()                          # detach: treat prev delta as input, not backprop through it

        return torch.stack(deltas, dim=1)                          # (N_paths, N_steps)

    def calc_pnl(self, S, deltas, initial_portfolio):
        """
        S : (N_paths, N_steps + 1)
        deltas : (N_paths, N_steps)
        """
        N_paths, N_steps = deltas.shape
        dt = self.T / N_steps

        ttm = torch.linspace(self.T, 0, S.shape[1])
        compound = torch.exp(self.r * (self.T - ttm[1:]))
        dS = S[:, 1:] - S[:, :-1] * torch.exp(torch.tensor(self.r * dt))
        hedging_pnl = torch.sum(deltas * dS * compound, dim=1)

        # transaction costs: κ * |Δδ_t| * S_t, compounded to maturity
        prev_deltas = torch.cat([torch.zeros(N_paths, 1), deltas[:, :-1]], dim=1)
        change_deltas = torch.abs(deltas - prev_deltas)
        transaction_costs = self.kappa * torch.sum(change_deltas * S[:, :-1] * compound, dim=1)

        initial_value_maturity = initial_portfolio * torch.exp(torch.tensor(self.r * self.T))
        payoff = torch.relu(S[:, -1] - self.K)

        pnl = initial_value_maturity + hedging_pnl - payoff - transaction_costs
        return pnl, transaction_costs

    # --- Risk measures ---

    def cvar_loss(self, pnl, alpha=0.95):
        losses = -pnl
        var = torch.quantile(losses, alpha).detach()
        return var + (1 / (1 - alpha)) * torch.mean(torch.relu(losses - var))

    def mse_loss(self, pnl):
        return torch.mean(pnl ** 2)

    def smse_loss(self, pnl):
        losses = -pnl[pnl < 0]
        if losses.numel() == 0:
            return torch.tensor(0.0)
        return torch.mean(losses ** 2)

    def variance_loss(self, pnl):
        return torch.var(pnl)

    # --- Training ---

    def train_deep_hedger(self, S, initial_portfolio, risk='cvar', alpha=0.95,
                          lr=1e-3, n_epochs=500):
        """Train on simulated paths S (numpy array)."""
        paths = torch.tensor(S, dtype=torch.float32)
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)

        print(f"\nTraining | risk={risk} | alpha={alpha} | kappa={self.kappa} | hidden={self.hidden_size} | lr={lr} | epochs={n_epochs}")
        for epoch in range(n_epochs + 1):
            optimizer.zero_grad()
            deltas = self.compute_deltas(paths)                    # sequential: uses δ_{t-1} as feature
            pnl, tc = self.calc_pnl(paths, deltas, initial_portfolio)

            if risk == 'cvar':
                loss = self.cvar_loss(pnl, alpha)
            elif risk == 'mse':
                loss = self.mse_loss(pnl)
            elif risk == 'smse':
                loss = self.smse_loss(pnl)
            elif risk == 'variance':
                loss = self.variance_loss(pnl)
            else:
                raise ValueError(f"Unknown risk measure: {risk}")

            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                print(f"Epoch {epoch:4d} | loss = {loss.item():.4f} | mean P&L = {pnl.mean().item():.4f} | std = {pnl.std().item():.4f} | mean TC = {tc.mean().item():.4f}")

    # --- Testing ---

    def test_deephedging(self, S, initial_portfolio, risk='cvar', alpha=0.95):
        """Evaluate trained model on out-of-sample paths S (numpy array)."""
        S_test = torch.tensor(S, dtype=torch.float32)
        N_paths, N_steps_plus1 = S_test.shape
        N_steps = N_steps_plus1 - 1

        self.eval()
        with torch.no_grad():
            deltas_test = self.compute_deltas(S_test)              # sequential
            pnl_test, tc_test = self.calc_pnl(S_test, deltas_test, initial_portfolio)

        if risk == 'cvar':
            loss_val = self.cvar_loss(pnl_test, alpha).item()
        elif risk == 'mse':
            loss_val = self.mse_loss(pnl_test).item()
        elif risk == 'smse':
            loss_val = self.smse_loss(pnl_test).item()
        elif risk == 'variance':
            loss_val = self.variance_loss(pnl_test).item()
        else:
            raise ValueError(f"Unknown risk measure: {risk}")

        print(f"\nTest | risk={risk} | loss={loss_val:.4f} | mean P&L={pnl_test.mean().item():.4f} | std={pnl_test.std().item():.4f} | mean TC={tc_test.mean().item():.4f}")
        return deltas_test, pnl_test

In [23]:
def BS_paths(S0, r, sigma, T, N_steps, M_paths, seed=123):
    np.random.seed(seed)
    dt = T / N_steps
    Z = np.random.randn(M_paths, N_steps)
    log_return = (r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
    log_S = np.log(S0) + np.cumsum(log_return, axis=1)
    S = np.exp(log_S)
    S = np.column_stack([np.full(M_paths, S0), S])
    return S.astype(np.float32)

# Parameters
S0, K, r, sigma, T = 100, 100, 0.0, 0.1, 1
initial_portfolio = float(S0 * norm.cdf((np.log(S0/K) + 0.5*sigma**2*T) / (sigma*np.sqrt(T)))
                          - K * np.exp(-r*T) * norm.cdf((np.log(S0/K) - 0.5*sigma**2*T) / (sigma*np.sqrt(T))))
print(f"Call premium: {initial_portfolio:.4f}")

# Simulate paths
S_train = BS_paths(S0, r, sigma, T, N_steps=52, M_paths=5000, seed=123)
S_test  = BS_paths(S0, r, sigma, T, N_steps=52, M_paths=1000, seed=456)

# --- Without transaction costs (kappa=0.0) ---

# Train
model = DeepHedging(K=K, r=r, T=T, kappa=0.0)
model.train_deep_hedger(S_train, initial_portfolio, risk='mse', alpha=0.95, lr=1e-3, n_epochs=500)

# Test
deltas, pnl = model.test_deephedging(S_test, initial_portfolio, risk='mse', alpha=0.95)

Call premium: 3.9878

Training | risk=mse | alpha=0.95 | kappa=0.0 | hidden=64 | lr=0.001 | epochs=500


NameError: name 'bs_delta' is not defined

In [ ]:
# --- With transaction costs (kappa=0.001) ---
# Train
model = DeepHedging(K=K, r=r, T=T, kappa=0.001)
model.train_deep_hedger(S_train, initial_portfolio, risk='mse', alpha=0.95, lr=1e-3, n_epochs=500)

# Test
deltas, pnl = model.test_deephedging(S_test, initial_portfolio, risk='mse', alpha=0.95)